# spark initiation

In [ ]:
!pip install pyspark -q
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
spark = (
    SparkSession.builder
    .appName("BT4221_HDB_Resale_Prices_Project")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)

spark.sparkContext.setLogLevel("WARN")

# Load data set

Modeled from DataGovSG sample code

In [ ]:
import os
import time
from pathlib import Path

import requests
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)


def load_secret(name: str, *alt_names: str) -> str:
    """Colab Secrets, or environment / `.env` when running locally."""
    try:
        from google.colab import userdata

        val = userdata.get(name)
        if val:
            return val
    except ImportError:
        pass
    try:
        from dotenv import load_dotenv

        load_dotenv(Path.cwd() / ".env")
    except ImportError:
        pass
    for key in (name, *alt_names):
        v = os.environ.get(key)
        if v:
            return v
    raise ValueError(
        f"Missing secret {name!r}. "
        "On Colab: add it under Secrets. Locally: set the env var or add it to `.env`."
    )


API_KEY = load_secret("GOV_DATA")

PRICE_DATASETS = [
    {"id": "d_8b84c4ee58e3cfc0ece0d773c8ca6abc", "label": "2017-present"},
    {"id": "d_ea9ed51da2787afaf8e51f827c304208", "label": "2015-2016"},
    {"id": "d_2d5ff9ea31397b66239f245f57751537", "label": "Mar2012-2014"},
    {"id": "d_43f493c6c50d54243cc1eab0df142d6a", "label": "2000-Feb2012"},
]

# fixed schema
PRICE_SCHEMA = StructType([
    StructField("month",               StringType(),  True),  # YYYY-MM
    StructField("town",                StringType(),  True),
    StructField("flat_type",           StringType(),  True),
    StructField("block",               StringType(),  True),
    StructField("street_name",         StringType(),  True),
    StructField("storey_range",        StringType(),  True),  # e.g. "07 TO 09"
    StructField("floor_area_sqm",      DoubleType(),  True),
    StructField("flat_model",          StringType(),  True),
    StructField("lease_commence_date", IntegerType(), True),  # YYYY
    StructField("remaining_lease",     StringType(),  True),  # e.g. "61 years 04 months"
    StructField("resale_price",        DoubleType(),  True),
    StructField("_source",             StringType(),  True),
])

def align_to_schema(df, label):
    """
    Align any dataset to the canonical schema regardless of which
    columns are present. Missing columns are added as null.
    Casts are applied after alignment so types are always consistent.
    """

    # Normalise column names to lowercase + underscores to match schema
    for col in df.columns:
        clean = col.strip().lower().replace(" ", "_")
        if clean != col:
            df = df.withColumnRenamed(col, clean)

    # Add missing columns as null
    canonical_cols = [
        "month", "town", "flat_type", "block", "street_name",
        "storey_range", "floor_area_sqm", "flat_model",
        "lease_commence_date", "remaining_lease", "resale_price"
    ]
    for col in canonical_cols:
        if col not in df.columns:
            print(f"    [{label}] '{col}' not found — adding as null")
            df = df.withColumn(col, F.lit(None).cast(StringType()))

    # Cast numeric columns safely after all columns are present
    df = (
        df
        .withColumn("floor_area_sqm",
                    F.when(F.col("floor_area_sqm").rlike(r"^-?\d+(\.\d+)?$"),
                           F.col("floor_area_sqm").cast(DoubleType()))
                    .otherwise(F.lit(None).cast(DoubleType())))
        .withColumn("lease_commence_date",
                    F.when(F.col("lease_commence_date").rlike(r"^\d+$"),
                           F.col("lease_commence_date").cast(IntegerType()))
                    .otherwise(F.lit(None).cast(IntegerType())))
        .withColumn("resale_price",
                    F.when(F.col("resale_price").rlike(r"^-?\d+(\.\d+)?$"),
                           F.col("resale_price").cast(DoubleType()))
                    .otherwise(F.lit(None).cast(DoubleType())))
    )

    # Keep only canonical columns + source tag in consistent order
    df = df.select(canonical_cols).withColumn("_source", F.lit(label))
    return df


s = requests.Session()
s.headers.update({
    "referer":   "https://colab.research.google.com",
    "x-api-key": API_KEY,
})

def download_dataset(dataset_id, label, schema, max_polls=10):
    print(f"  [{label}]  initiating download...")

    s.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download",
        json={}
    )

    for i in range(max_polls):
        poll = s.get(
            f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download",
            json={}
        ).json()

        if "url" in poll["data"]:
            url = poll["data"]["url"]
            print(f"    ready, downloading...")

            lines = requests.get(url).text.splitlines()
            rdd   = spark.sparkContext.parallelize(lines)
            df    = (spark.read
                         .option("header", "true")
                         .csv(rdd))
            df = df.withColumn("_source", F.lit(label))

            # Align to canonical schema — adds missing columns as null,
            # casts existing columns to correct types
            df = align_to_schema(df, label)
            print(f"    done ✓  ({df.count():,} rows)")
            return df

        print(f"    poll {i+1}/{max_polls}: not ready yet...")
        time.sleep(3)

    raise TimeoutError(f"Dataset {label} never became ready after {max_polls} polls")

def load_from_api(dfs, schema):
    dfs = [download_dataset(ds["id"], ds["label"], schema) for ds in dfs]

    # Union all 4 DataFrames
    from functools import reduce
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

    print(f"\n  Total rows loaded: {combined.count():,}")
    return combined

In [ ]:
prices_df = load_from_api(PRICE_DATASETS,PRICE_SCHEMA)

In [ ]:
prices_df.printSchema()

In [ ]:
prices_df.filter(F.col("_source") == "2000-Feb2012").show(10)

# cleaning agent





In [ ]:
!pip install -q langgraph langchain langchain-openai matplotlib seaborn

In [ ]:
import json
import openai
from pyspark.sql.types import (
    StringType, LongType,
    FloatType, DateType, TimestampType
)
NUMERIC_TYPES = (DoubleType, FloatType, IntegerType, LongType)

In [ ]:
client = openai.OpenAI(api_key=load_secret("BT4221_OPENAI_API_KEY", "OPENAI_API_KEY"))
MAX_RETRIES = 2

CLEANING TOOLS

In [ ]:
def _transaction_month_to_date(column_name):
    """
    Parse a transaction-month column whether values are YYYY-MM, YYYY-MM-DD,
    or already Date/Timestamp. Avoids appending '-01' to a full date (which
    yields invalid strings like '2017-01-01-01').
    """
    s = F.trim(F.col(column_name).cast("string"))
    return (
        F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(s, "yyyy-MM-dd"))
        .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd"))
        .otherwise(F.try_to_date(s))
    )


def tool_trim_uppercase(sdf, column, **kwargs):
    """Trim whitespace and upper-case a string column."""
    return sdf.withColumn(column, F.upper(F.trim(F.col(column))))

def tool_parse_yyyymm_date(sdf, column, **kwargs):
    """
    Parse a YYYY-MM (or YYYY-MM-DD) column into DateType (first of month).
    Replaces the original column in-place.
    """
    return sdf.withColumn(column, _transaction_month_to_date(column))

def tool_parse_yyyymmdd_date(sdf, column, **kwargs):
    """Parse a YYYY-MM-DD string column into a DateType column."""
    return sdf.withColumn(column, F.to_date(F.col(column), "yyyy-MM-dd"))

def tool_parse_lease_string(sdf, column, **kwargs):
    """
    Parse remaining_lease string → numeric years, replacing the original column.
    Handles: '61 years 04 months', '62 years 01 month', '63 years'
    """
    years = F.nullif(
        F.regexp_extract(F.col(column), r"(\d+)\s*[Yy]ear", 1), F.lit("")
    ).cast(DoubleType())

    months = F.coalesce(
        F.nullif(F.regexp_extract(F.col(column), r"(\d+)\s*[Mm]onth", 1), F.lit(""))
         .cast(DoubleType()),
        F.lit(0.0)
    )
    # Replace original column
    return sdf.withColumn(column, F.coalesce(years, F.lit(None).cast(DoubleType())) + months / 12.0)

def tool_drop_nulls(sdf, column, **kwargs):
    """Drop rows where the specified column is null or empty string."""
    return sdf.filter(
        F.col(column).isNotNull() & (F.trim(F.col(column).cast("string")) != "")
    )

def tool_flag_outliers_zscore(sdf, column, group_by=None, threshold=4.0, **kwargs):
    """
    Add a boolean column {column}_outlier flagging rows where the z-score
    of `column` exceeds `threshold` (default 4.0).
    If group_by is provided, z-score is computed within each group.
    Rows are flagged, NOT removed.
    """
    out_col = column + "_outlier"
    w = Window.partitionBy(group_by) if group_by else Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )
    return (
        sdf
        .withColumn("_mean", F.mean(column).over(w))
        .withColumn("_std",  F.stddev(column).over(w))
        .withColumn(
            out_col,
            F.when(
                F.col("_std") > 0,
                F.abs(F.col(column) - F.col("_mean")) / F.col("_std") > threshold
            ).otherwise(F.lit(False))
        )
        .drop("_mean", "_std")
    )

def tool_standardise_values(sdf, column, replacements, **kwargs):
    """
    Replace inconsistent string values in a column.
    `replacements` is a dict: {"old_value": "new_value", ...}
    e.g. {"MULTI GENERATION": "MULTI-GENERATION", "MULTI-GEN": "MULTI-GENERATION"}
    """
    for old, new in replacements.items():
        sdf = sdf.withColumn(
            column,
            F.when(F.col(column) == old, F.lit(new)).otherwise(F.col(column))
        )
    return sdf

def tool_impute_remaining_lease(sdf, column, **kwargs):
    """
    Impute missing remaining_lease values using:
        99 - (transaction_year - lease_commence_date)
    Only fills nulls — existing values are preserved.
    Requires: lease_commence_date (int) and month (YYYY-MM string) to be present.
    """
    sale_year = F.year(_transaction_month_to_date("month"))
    approximated = (99 - (sale_year - F.col("lease_commence_date"))).cast(DoubleType())

    return sdf.withColumn(
        column,
        F.coalesce(F.col(column), approximated)   # keep existing, fill nulls
    )
def tool_parse_storey_range(sdf, column, **kwargs):
    """
    Parse 'XX TO YY' storey_range string into a numeric storey_midpoint column.
    e.g. "04 TO 06" → storey_midpoint = 5.0
    """
    lo = F.regexp_extract(F.col(column), r"^(\d+)", 1).cast("double")
    hi = F.regexp_extract(F.col(column), r"(\d+)$", 1).cast("double")
    return sdf.withColumn("storey_midpoint", (lo + hi) / 2)


def tool_derive_year_month(sdf, column, **kwargs):
    """
    Extract transaction_year (int) and transaction_month (int) from a YYYY-MM column.
    Requires the column to be either a string 'YYYY-MM' or already a DateType.
    """
    date_col = _transaction_month_to_date(column)
    return (sdf
        .withColumn("transaction_year",  F.year(date_col).cast("int"))
        .withColumn("transaction_month", F.month(date_col).cast("int"))
    )


def tool_flag_outliers_iqr(sdf, column, multiplier=1.5, **kwargs):
    """
    Flag outliers using the IQR method. Adds a boolean column {column}_outlier.
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    Rows are flagged, NOT removed here (removal happens in post-processing).
    """
    quantiles = sdf.approxQuantile(column, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    out_col = f"{column}_outlier"
    return sdf.withColumn(
        out_col,
        (F.col(column) < lower) | (F.col(column) > upper)
    )


def tool_compute_remaining_lease(sdf, column, **kwargs):
    """
    Always compute remaining_lease_years from lease_commence_date.
    Formula: 99 - (transaction_year - lease_commence_date)
    Overwrites the column for ALL rows — no string parsing needed.
    Reliable for all 4 datasets including pre-2015 data where remaining_lease is null.
    Requires: transaction_year (int) and lease_commence_date (int) columns.
    Run AFTER derive_year_month.
    """
    computed = (99 - (F.col("transaction_year") - F.col("lease_commence_date"))).cast(DoubleType())
    return sdf.withColumn("remaining_lease_years", computed)


def tool_drop_column(sdf, column, **kwargs):
    """Drop a column nominated by the agent in columns_to_drop."""
    return sdf.drop(column)


In [ ]:
# Registry — maps tool name (string) → function
TOOL_REGISTRY = {
    "trim_uppercase":         tool_trim_uppercase,
    "parse_yyyymm_date":      tool_parse_yyyymm_date,
    "parse_yyyymmdd_date":    tool_parse_yyyymmdd_date,
    "parse_lease_string":     tool_parse_lease_string,
    "drop_nulls":             tool_drop_nulls,
    "flag_outliers_zscore":   tool_flag_outliers_zscore,
    "flag_outliers_iqr":      tool_flag_outliers_iqr,
    "standardise_values":     tool_standardise_values,
    "impute_remaining_lease": tool_impute_remaining_lease,
    "parse_storey_range":     tool_parse_storey_range,
    "derive_year_month":      tool_derive_year_month,
    "compute_remaining_lease": tool_compute_remaining_lease,
    "drop_column":             tool_drop_column,
}

TOOL_CATALOGUE = """
Available cleaning tools (name → what it does → required args → optional args):
  trim_uppercase          → trim whitespace and upper-case a string column
                            args: column
  parse_yyyymm_date       → parse YYYY-MM string → DateType (first of month)
                            args: column
  parse_yyyymmdd_date     → parse YYYY-MM-DD string → DateType
                            args: column
  parse_lease_string      → extract years from '61 years 04 months' or plain '61'
                            → replaces column with double (years)
                            args: column
  drop_nulls              → drop rows where column is null or empty
                            args: column
  flag_outliers_iqr       → PREFERRED outlier method: add {col}_outlier boolean using IQR
                            lower = Q1 - multiplier*IQR, upper = Q3 + multiplier*IQR
                            args: column   optional: multiplier (float, default 1.5)
                            USE THIS for resale_price (required by project spec)
  flag_outliers_zscore    → add {col}_outlier boolean (z-score > threshold, default 4.0)
                            args: column   optional: group_by (column name), threshold (float)
  standardise_values      → replace inconsistent string values
                            args: column, replacements (dict of old→new)
  impute_remaining_lease  → fill null remaining_lease values using
                            99 - (sale_year - lease_commence_date)
                            args: column
                            requires: month and lease_commence_date columns
  parse_storey_range      → parse 'XX TO YY' string → numeric storey_midpoint column
                            args: column (must be storey_range)
  derive_year_month       → extract transaction_year (int) and transaction_month (int)
                            from a YYYY-MM column
                            args: column (must be month)
  compute_remaining_lease → always compute remaining_lease_years = 99 - (transaction_year - lease_commence_date)
                            args: column (ignored — always writes remaining_lease_years)
                            requires: transaction_year and lease_commence_date columns
                            run AFTER derive_year_month
  drop_column            → drop a column nominated by the agent (columns_to_drop list)
                            args: column
"""

PROFILER

dataset info for llm to plan

In [ ]:
def profile_dataframe(sdf):
    total       = sdf.count()
    sample_rows = sdf.limit(3).collect()

    # Header
    lines = [
        f"Total rows: {total:,}",
        "",
        f"{'Column':<25} {'Type':<15} {'Nulls':>8} {'Distinct':>10}  {'Patterns':<30}  {'Samples'}",
        "-" * 110
    ]

    for field in sdf.schema.fields:
        col   = field.name
        dtype = str(field.dataType)

        # Nulls
        if isinstance(field.dataType, StringType):
            null_n = sdf.filter(F.col(col).isNull() | (F.trim(F.col(col)) == "")).count()
        else:
            null_n = sdf.filter(F.col(col).isNull()).count()
        null_str = f"{null_n/total*100:.0f}%" if null_n > 0 else "0%"

        # Distinct
        distinct_n = sdf.select(col).distinct().count()

        # Patterns (string only)
        patterns = ""
        if isinstance(field.dataType, StringType):
            non_null   = sdf.filter(F.col(col).isNotNull() & (F.trim(F.col(col)) != ""))
            non_null_n = non_null.count()
            if non_null_n > 0:
                def pct(pattern):
                    return non_null.filter(F.col(col).rlike(pattern)).count() / non_null_n * 100
                checks = [
                    ("YYYY-MM",   pct(r"^\d{4}-\d{2}$")),
                    ("X TO Y",    pct(r"^\d+ TO \d+$")),
                    ("unit str",  pct(r"\d+\s*(year|month)")),
                ]
                patterns = "  ".join(f"{label}:{v:.0f}%" for label, v in checks if v > 10)

        # Numeric stats
        if isinstance(field.dataType, NUMERIC_TYPES):
            s = sdf.select(F.min(col), F.max(col)).collect()[0]
            patterns = f"min={s[0]}  max={s[1]}"

        # Samples
        samples = ", ".join(str(r[col]) for r in sample_rows if r[col] is not None)[:50]

        lines.append(
            f"{col:<25} {dtype:<15} {null_str:>8} {distinct_n:>10}  {patterns:<30}  {samples}"
        )

    return "\n".join(lines)

LLM helper

In [ ]:
def ask_llm_for_config(profile_str):
    """
    Ask the LLM to return a structured cleaning_config.
    The LLM decides the strategy; config_to_plan() translates it to tool calls,
    ensuring PySpark cleaning is driven by the agent's config — not hardcoded.
    """
    prompt = f"""You are a data cleaning expert for an HDB resale price ML pipeline.
Analyse the DataFrame profile and return a cleaning configuration JSON.

{TOOL_CATALOGUE}

DATAFRAME PROFILE:
{profile_str}

Return ONLY valid JSON with exactly these six keys:

{{
  "storey_range_strategy": "parse_storey_range",
  "outlier_method": "IQR",
  "outlier_threshold": <float, choose 1.5 to 3.0 based on price spread>,
  "null_handling": {{
    "<column_name>": "drop_nulls" | "impute_remaining_lease"
  }},
  "columns_to_drop": ["<col>"],
  "duplicate_strategy": "dropDuplicates"
}}

Rules:
  - storey_range_strategy: always "parse_storey_range"
  - outlier_method: always "IQR" (required by project spec)
  - outlier_threshold: between 1.5 and 3.0 — choose based on the resale_price spread
  - null_handling: only include columns that have >0 nulls in the profile;
    use "impute_remaining_lease" for remaining_lease, "drop_nulls" for others
  - columns_to_drop: list columns that add no predictive value (e.g. _source)
  - duplicate_strategy: always "dropDuplicates"
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


def config_to_plan(cleaning_config):
    """
    Convert a cleaning_config dict into an ordered list of
    tool calls compatible with TOOL_REGISTRY.
    This is how PySpark cleaning is driven by the agent's config.
    """
    plan = []

    # 1. Trim + uppercase all string categoricals
    for col in ["town", "flat_type", "flat_model", "street_name", "block"]:
        plan.append({"tool": "trim_uppercase", "column": col, "kwargs": {},
                     "reason": "Normalise categorical text"})

    # 2. Parse month -> DateType, then extract integer year/month
    plan.append({"tool": "parse_yyyymm_date", "column": "month", "kwargs": {},
                 "reason": "Convert YYYY-MM string to DateType"})
    plan.append({"tool": "derive_year_month", "column": "month", "kwargs": {},
                 "reason": "Extract transaction_year and transaction_month"})

    # 3. Storey range -> midpoint (driven by config)
    if cleaning_config.get("storey_range_strategy") == "parse_storey_range":
        plan.append({"tool": "parse_storey_range", "column": "storey_range", "kwargs": {},
                     "reason": "Parse storey_range string to numeric storey_midpoint"})

    # 4. Flat type standardisation (canonical values)
    plan.append({
        "tool": "standardise_values", "column": "flat_type",
        "kwargs": {"replacements": {
            "MULTI GENERATION": "MULTI-GENERATION",
            "MULTI-GEN": "MULTI-GENERATION",
            "multi generation": "MULTI-GENERATION",
        }},
        "reason": "Standardise flat_type to canonical values"
    })

    # 5. Compute remaining_lease_years from lease_commence_date (reliable for all 4 datasets)
    # Replaces parse_lease_string + impute_remaining_lease — handles null columns for pre-2015 data
    plan.append({"tool": "compute_remaining_lease", "column": "remaining_lease", "kwargs": {},
                 "reason": "Compute remaining_lease_years = 99 - (transaction_year - lease_commence_date) for all rows"})
    # Drop original remaining_lease string column — always superseded by remaining_lease_years
    plan.append({"tool": "drop_column", "column": "remaining_lease", "kwargs": {},
                 "reason": "Drop original remaining_lease string column (replaced by remaining_lease_years)"})

    # 6. Null handling per config (agent decides which columns need drop_nulls)
    for col, strategy in cleaning_config.get("null_handling", {}).items():
        if strategy == "drop_nulls" and col != "remaining_lease":
            plan.append({"tool": "drop_nulls", "column": col, "kwargs": {},
                         "reason": f"Remove null rows in {col} (agent decision from cleaning_config)"})

    # 6b. Drop columns nominated by agent (columns_to_drop)
    for col in cleaning_config.get("columns_to_drop", []):
        plan.append({"tool": "drop_column", "column": col, "kwargs": {},
                     "reason": f"Drop {col} per agent config — no predictive value"})

    # 7. Outlier flagging driven by config method + threshold
    method    = cleaning_config.get("outlier_method", "IQR")
    threshold = float(cleaning_config.get("outlier_threshold", 1.5))
    if method == "IQR":
        plan.append({"tool": "flag_outliers_iqr", "column": "resale_price",
                     "kwargs": {"multiplier": threshold},
                     "reason": f"Flag resale_price outliers using IQR (multiplier={threshold}) per agent config"})
    else:
        plan.append({"tool": "flag_outliers_zscore", "column": "resale_price",
                     "kwargs": {"threshold": threshold},
                     "reason": f"Flag resale_price outliers using z-score (threshold={threshold}) per agent config"})

    return plan


def ask_llm_to_fix(error_type, error_msg, failed_call, profile_str):
    prompt = f"""A cleaning tool call has failed. Revise the arguments to fix it.

FAILED CALL:
{json.dumps(failed_call, indent=2)}

ERROR TYPE   : {error_type}
ERROR MESSAGE: {error_msg}

{TOOL_CATALOGUE}

DATAFRAME PROFILE:
{profile_str}

Rules:
  - Revise only the arguments of the failed call — do not change the intent
  - If the column type is wrong for the tool, suggest the correct tool instead
  - Do not suggest null handling tools — those are handled separately by the human
  - Do not suggest impute_remaining_lease unless the original call was already
    related to remaining_lease

Return ONLY valid JSON, no markdown:
{{
  "diagnosis": "one sentence root cause",
  "revised_call": {{
    "tool": "tool_name",
    "column": "column_name",
    "kwargs": {{}}
  }}
}}"""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


PIPELINE

In [ ]:
print("\nStep 1: Profiling DataFrame...")
profile_str = profile_dataframe(prices_df)
print(profile_str)

# Null counts BEFORE cleaning
print("\n=== Null counts BEFORE cleaning ===")
prices_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in prices_df.columns]
).show(vertical=True)

print("\nStep 2: Asking LLM for cleaning configuration...")
cleaning_config = ask_llm_for_config(profile_str)
print("\nCleaning config (agent decision):")
print(json.dumps(cleaning_config, indent=2))

# Save cleaning_decisions.md immediately after LLM returns config
import datetime
with open("cleaning_decisions.md", "w") as f:
    f.write("# Cleaning Decisions\n\n")
    f.write(f"**Agent:** Data Cleaning Agent (gpt-4o-mini)\n")
    f.write(f"**Timestamp:** {datetime.datetime.now().isoformat()}\n\n")
    f.write("## Cleaning Config\n\n")
    f.write(f"```json\n{json.dumps(cleaning_config, indent=2)}\n```\n")
print("\ncleaning_decisions.md written ✓")

# Convert config to ordered tool-call plan (PySpark reads config via this function)
plan = config_to_plan(cleaning_config)
print(f"\nDerived plan — {len(plan)} tool calls:")
for i, call in enumerate(plan, 1):
    kwargs_str = f"  kwargs={call['kwargs']}" if call.get("kwargs") else ""
    print(f"  {i}. {call['tool']}({call['column']}){kwargs_str}")
    print(f"     → {call['reason']}")

fix_log     = []
current_sdf = prices_df

print("\nStep 3: Executing plan...")
print("=" * 52)

for call in plan:
    tool_name    = call["tool"]
    column       = call["column"]
    kwargs       = call.get("kwargs", {})
    current_call = call.copy()

    print(f"\n  {tool_name}({column})")

    if tool_name not in TOOL_REGISTRY:
        print(f"  SKIPPED - unknown tool '{tool_name}'")
        continue

    success = False

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            result_sdf = TOOL_REGISTRY[tool_name](current_sdf, column, **kwargs)
            result_sdf.count()   # force Spark evaluation

            print(f"  Attempt {attempt}: PASSED")
            current_sdf = result_sdf
            success = True
            break

        except Exception as e:
            err_type = type(e).__name__
            err_msg  = str(e)[:300]
            print(f"  Attempt {attempt}: FAILED  ({err_type}: {err_msg[:80]})")

            if attempt > MAX_RETRIES:
                print("  Max retries reached. Skipping.")
                break

            print("  Asking LLM to revise call...")
            llm_result   = ask_llm_to_fix(err_type, err_msg, current_call, profile_str)
            diagnosis    = llm_result.get("diagnosis", "Unknown")
            revised      = llm_result.get("revised_call", {})
            tool_name    = revised.get("tool", tool_name)
            column       = revised.get("column", column)
            kwargs       = revised.get("kwargs", kwargs)
            current_call = revised

            print(f"  Diagnosis    : {diagnosis}")
            print(f"  Revised call : {tool_name}({column}) kwargs={kwargs}")

            fix_log.append({
                "original": call,
                "attempt":  attempt,
                "error":    err_msg[:150],
                "diagnosis": diagnosis,
                "revised":  revised,
            })

    if not success:
        print(f"  Could not complete after {MAX_RETRIES} retries.")


In [ ]:
print("PIPELINE FINISHED")
print(f"Total automatic revisions: {len(fix_log)}")

print("\nFINAL SCHEMA")
current_sdf.printSchema()

print("\nFINAL SAMPLE")
current_sdf.show(5, truncate=False)

if fix_log:
    print("\nFIX LOG")
    print("-" * 52)
    for i, fix in enumerate(fix_log, 1):
        orig = fix["original"]
        print(f"\nRevision {i} — {orig['tool']}({orig['column']})  attempt {fix['attempt']}")
        print(f"  Error    : {fix['error']}")
        print(f"  Diagnosis: {fix['diagnosis']}")
        print(f"  Revised  : {fix['revised']}")

clean_df = current_sdf
print(f"\nclean_df ready: {clean_df.count():,} rows, {len(clean_df.columns)} columns")

# ── POST-PROCESSING (deterministic) ─────────────────────────────────────────

# 0. Deterministic flat_type whitelist validation
CANONICAL_FLAT_TYPES = {"1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"}
n_invalid = clean_df.filter(~F.col("flat_type").isin(list(CANONICAL_FLAT_TYPES))).count()
if n_invalid > 0:
    print(f"\nWarning: {n_invalid:,} rows with non-canonical flat_type — filtering out")
    clean_df = clean_df.filter(F.col("flat_type").isin(list(CANONICAL_FLAT_TYPES)))
else:
    print("flat_type: all values are canonical ✓")

# 1. Validate floor_area_sqm — no negative values
n_neg = clean_df.filter(F.col("floor_area_sqm") <= 0).count()
if n_neg > 0:
    print(f"Removing {n_neg:,} rows with non-positive floor_area_sqm")
    clean_df = clean_df.filter(F.col("floor_area_sqm") > 0)

# 2. Remove exact duplicates
n_before_dedup = clean_df.count()
clean_df = clean_df.dropDuplicates()
n_after_dedup = clean_df.count()
print(f"\nDuplicates removed: {n_before_dedup - n_after_dedup:,} rows ({n_before_dedup:,} → {n_after_dedup:,})")

# 3. Remove outliers flagged by IQR
outlier_col = "resale_price_outlier"
n_removed = 0
if outlier_col in clean_df.columns:
    n_before_out = clean_df.count()
    clean_df = clean_df.filter(~F.col(outlier_col)).drop(outlier_col)
    n_after_out = clean_df.count()
    n_removed = n_before_out - n_after_out
    print(f"Outliers removed  : {n_removed:,} rows ({n_removed/n_before_out*100:.1f}%) — IQR method on resale_price")

# 4. Null counts AFTER cleaning
print("\n=== Null counts AFTER cleaning ===")
clean_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in clean_df.columns]
).show(vertical=True)

# 5. Consolidated cleaning summary
n_raw = prices_df.count()  # actual raw row count
n_final = clean_df.count()
print("\n=== Cleaning Summary (Before vs After) ===")
print(f"{'Stage':<35} {'Rows':>10}  {'Change':>20}")
print("-" * 68)
print(f"{'Raw rows loaded':<35} {n_raw:>10,}  {'—':>20}")
print(f"{'After deduplication':<35} {n_after_dedup:>10,}  {f'-{n_before_dedup - n_after_dedup:,} dupes':>20}")
print(f"{'After outlier removal':<35} {n_final:>10,}  {f'-{n_removed:,} outliers':>20}")
print(f"{'Total rows removed':<35} {n_raw - n_final:>10,}  {f'{(n_raw - n_final)/n_raw*100:.1f}% of raw':>20}")
print(f"\nclean_df final: {n_final:,} rows, {len(clean_df.columns)} columns")


# EDA

In [ ]:
# ── EDA 1: Price trends over time ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

df = clean_df

trend_pdf = (
    df
      .groupBy("month", "flat_type")
      .agg(F.percentile_approx("resale_price", 0.5).alias("median_price"))
      .toPandas()
)

trend_pdf["month"] = pd.to_datetime(trend_pdf["month"])
trend_pdf = trend_pdf.sort_values("month")

trend_pdf["median_price"] = (
    trend_pdf.groupby("flat_type")["median_price"]
             .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

fig, ax = plt.subplots(figsize=(13,5))

for ft, grp in trend_pdf.groupby("flat_type"):
    ax.plot(grp["month"], grp["median_price"]/1e3, linewidth=2, label=ft)

import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_title("Median HDB Resale Price Trend by Flat Type", fontsize=14, fontweight="bold")
ax.set_ylabel("Median Price (SGD '000)")
ax.set_xlabel("Year")

ax.legend(title="Flat Type", bbox_to_anchor=(1.02,1))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 2: Price distribution by flat type & town ──
dist_pdf = (
    df
      .select("resale_price", "flat_type", "town")
      .toPandas()
)

# Violin by flat_type
fig, ax = plt.subplots(figsize=(12, 5))
order = dist_pdf.groupby("flat_type")["resale_price"].median().sort_values().index
sns.violinplot(data=dist_pdf, x="flat_type", y="resale_price",
               order=order, palette="muted", ax=ax, inner="quartile")
ax.set_title("Resale Price Distribution by Flat Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Flat Type")
ax.set_ylabel("Resale Price (SGD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# Box by top-10 towns
top10 = dist_pdf.groupby("town")["resale_price"].median().nlargest(10).index
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=dist_pdf[dist_pdf["town"].isin(top10)], x="town", y="resale_price",
            order=top10, palette="Set2", ax=ax, flierprops={"markersize": 2})
ax.set_title("Resale Price Distribution — Top 10 Towns by Median Price",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Town")
ax.set_ylabel("Resale Price (SGD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 3: Correlation heatmap ──
corr_cols = [c for c in [
    "resale_price", "floor_area_sqm", "lease_commence_date",
    "storey_midpoint", "transaction_year", "transaction_month", "remaining_lease_years"
] if c in df.columns]

corr_pdf = df.select(corr_cols).dropna().limit(50_000).toPandas()
corr_pdf = corr_pdf.select_dtypes(include="number")
corr_mat = corr_pdf.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n  Correlations with resale_price (absolute, descending):")
print(
    corr_mat["resale_price"]
    .drop("resale_price")
    .abs()
    .sort_values(ascending=False)
    .to_string()
)

End

In [ ]:
spark.stop()
print('Spark session stopped.')